# 00 — Data inventory and understanding

This notebook explains the corrected raw EMA and Garmin sources using **aggregates only**.
It does not calculate PSS scores, HRV, prediction features or a final linked dataset.
All exploratory results use the existing development participants; final-test participants
appear only in structural identity/split verification. Real rows, identifiers, filenames,
free-text answers and locations are never displayed.

Read `../README.md` for setup. Source notebook outputs must stay empty; save executed copies
only under ignored `outputs/`.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
assert (ROOT / "src/garmin_inventory").is_dir()
RUN_FULL_AUDIT = os.environ.get("GARMIN_RUN_FULL_AUDIT") == "1"
if RUN_FULL_AUDIT:
    env = dict(os.environ, PYTHONPATH=str(ROOT / "src"))
    result = subprocess.run([sys.executable, "-m", "garmin_inventory.audit", "--root", str(ROOT)],
                            cwd=ROOT, env=env, capture_output=True, text=True)
    # The CLI emits aggregate-only progress and sanitized failures.
    print(result.stdout)
    if result.returncode:
        raise RuntimeError("Raw audit stopped; review its sanitized status before continuing.")
else:
    print("Displaying saved aggregate results. Set GARMIN_RUN_FULL_AUDIT=1 to recompute from raw archives.")

report = json.loads((ROOT / "reports/inventory.json").read_text())
dictionary = json.loads((ROOT / "reports/data_dictionary.json").read_text())
assert report["verification"]["final_test_exploratory_rows"] == 0

## What is present

The outer archives contain corrected EMA CSVs, questionnaire PDFs, and nested Garmin ZIPs.
Archive members are read without extraction. Logical source accounting removes an exact export
copy only for diagnostics; the raw sources remain unchanged. The previous project's extra loose
copy is absent from these local archives. Legacy exports and model artifacts are not inputs.

In [ ]:
display(pd.DataFrame(report["archives"]))
display(pd.DataFrame(report["structural"]["ema_tables"]))
structural = report["structural"]
display(pd.DataFrame([{
    "EMA rows": structural["ema_rows"],
    "EMA participants": structural["ema_participants"],
    "Garmin CSV tables": structural["garmin_csv_tables"],
    "Garmin logical rows": structural["garmin"]["logical_rows"],
    "Garmin participants": structural["garmin"]["distinct_mapped_participants"],
}]))

## Linkage keys and the frozen split

Exact participant keys and content-verified respondent crosswalks support diagnostic matching.
One respondent key is ambiguous. Three raw source schemas lack identifier columns; exact tokens
in filenames are corroborated with previous normalized provenance and raw CSV content hashes.
This is reference consistency, not independent proof of identity inside keyless CSVs.

Participant + scheduled instance alone has a collision; adding survey category resolves the
observed EMA collision. Keep actual-start clocks separate from this event identifier.
The frozen assignment bytes and all existing development folds are preserved.

In [ ]:
display(pd.DataFrame([report["frozen_split"]]))
display(pd.DataFrame(report["structural"]["garmin"]["schemas"]))
display(pd.DataFrame([{"mapping_method": k, "rows": v}
                     for k, v in structural["garmin"]["mapping_methods_rows"].items()]))

## Fields, missingness and clocks

Every source field is documented. Missingness/type counts are development-only, and never
include response values. PSS export codes are 1–5, although questionnaire labels are 0–4.
The agreed three-item score would be `nervous + (6 - cope) + (6 - control)` (3–15), requiring
complete valid integer items. No score is computed here.

EMA offsets are minutes local-minus-UTC; scheduled UTC is checked against the instance epoch.
Actual starts are the prospective boundary. Retain the reference 60-second buffer for minute
precision and no additional buffer for second precision. Garmin `ts` is Unix seconds with acquisition-versus-receipt semantics unconfirmed; it is
not a scheduled EMA instance and receives no timezone shift. Naive text clocks are checked diagnostically, not repaired. Acquisition/upload semantics
are still unresolved, and no dedicated submission/end timestamp exists.

In [ ]:
fields = pd.DataFrame(dictionary)
display(fields.loc[fields["field"].isin([
    "mbl_cod", "rsp_id", "instance_id", "actual_start_local", "scheduled_start_local",
    "timezone_offset", "PSS_NERVOUS", "PSS_COPE", "PSS_CONTROL", "ts", "dte_tme", "val"
])])
if report["frozen_split"]["available"]:
    display(pd.DataFrame(report["pss_item_validity_development"]))
    display(pd.DataFrame([report["ema_clocks_development"]]))
    display(pd.DataFrame([report["garmin_clocks_development"]]))

## Recording quality and preceding history

The table of positive sampling intervals uses distinct timestamps within participant/sensor.
Continuous-run lengths split at >2-second gaps for HR/IBI and >60-second gaps for other
sensors. These are descriptive diagnostics, not validated eligibility rules. Calorie intervals
slightly above 60 seconds can split runs even at an approximately minute cadence. The HR result
of no five-minute runs under the 2-second rule does not establish absence of usable HR data;
HR usability needs a separately justified sensor-specific policy. Runs do not certify clean NN intervals. A long first-to-last span is not continuous wear. Zeros, gaps, resets, duplicate
records and conflicting values are distinct diagnostics. Quality diagnostics are not initial
predictors. The report retains all records and makes no imputation.

History intervals are `[cutoff - lookback, cutoff)`. The denominator includes every development
EMA, including people without recordings. One observation is weaker evidence than bounded gaps,
and neither is an approved HRV eligibility criterion. Long lookbacks will summarize valid short
windows later; they will not require or imply a continuous 24-hour HRV recording.

In [ ]:
if report["frozen_split"]["available"]:
    sensors = pd.DataFrame(report["sensors_development"])
    display(sensors[["sensor", "rows", "zero_values", "outside_provisional_range",
                     "duplicate_timestamp_excess", "positive_distinct_interval_median_seconds",
                     "gaps_above_3600s", "run_gap_threshold_seconds", "longest_continuity_run_seconds"]])
    history_table = pd.DataFrame(report["history_development"])
    display(history_table.loc[history_table.sensor == "ibi"])
    display(pd.DataFrame([report["ibi_clock_diagnostic_development"]]))

## Synthetic conceptual rows — not calculations

- Sensor: `SYNTHETIC_A`, IBI, `2030-01-01T11:57:00.500Z`, `800 ms`, source-row provenance.
- Five-minute window: for a second-resolution actual start at `12:00:00Z`,
  `[11:55:00Z, 12:00:00Z)`, with `eligibility`, `RMSSD` and `SDNN` **not calculated**.
  A minute-resolution start recorded at `12:00` uses cutoff `11:59:00Z`.
- EMA-linked row: synthetic event/category, actual start, cutoff, frozen fold,
  PSS score and `rmssd_15m_mean` **not calculated**. Counts/gaps stay in a separate quality table.

Planned lookback summaries preserve the feature-sheet list specified for this checkpoint:
**mean, SD, minimum, maximum, trend and latest**. **Median and IQR are additional candidates**,
not replacements. Trend is a slope against actual window times, with settings to be specified later.

Candidate families are interval HRV, HR summaries, validated activity increments and possibly
PulseOx. Garmin stress and its derivatives remain exploratory. Quality/coverage diagnostics and
VLF-related quantities are excluded from the initial predictors.

NeuroKit2 accepts `RRI` (milliseconds) and `RRI_Time` (seconds). HeartPy `process_rr` accepts
millisecond intervals but no independent timestamp vector, so gap eligibility comes first.
Package HRV indices and custom lookback summaries are different operations; settings and versions
must be recorded. See the linked official documentation and detailed decisions in the plan.

In [ ]:
display(Markdown((ROOT / "reports/feature_pipeline_plan.md").read_text()))

## Verification and next step

The audit checks raw/reference hashes and modification times, source accounting, exact split
coverage, development-only exploration and known private tokens in public artifacts. Synthetic
tests cover timing boundaries, timezone conversion, ambiguous keys, archive traversal, duplicate
exports, test-set isolation and public-output protections.

**Next:** implement raw-preserving preprocessing and a development-only five-minute IBI eligibility
audit with one-minute steps. Resolve timestamp meaning and quality policy before HRV calculation.
The full findings are in `reports/inventory_report.md`, `reports/data_dictionary.md` and
`reports/feature_pipeline_plan.md`.

In [ ]:
display(pd.DataFrame([report["verification"]]))